# ToneFit ML — Full Pipeline (Google Colab)

Run this notebook top to bottom to execute the entire ToneFit pipeline.

**Before you start:**
- Go to `Runtime → Change runtime type → T4 GPU` (free, makes deep learning much faster)

**Pipeline steps in this notebook:**
1. Setup — clone repo, install dependencies, enable GPU
2. Data Collection — scrape celebrity face images
3. Preprocessing — extract color features, split dataset
4. EDA — visualize the dataset
5. Train Traditional ML — SVM + Random Forest
6. Train Deep Learning — MobileNetV2
7. Evaluate — compare all 3 models
8. Predict — test on a photo

---
## Step 0 — Check GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU available: {gpus[0].name}")
    print("You're good to go!")
else:
    print("WARNING: No GPU detected.")
    print("Deep learning training will be slow.")
    print("Go to Runtime → Change runtime type → T4 GPU, then re-run.")

---
## Step 1 — Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL = "https://github.com/ajipal/ToneFit.git"
REPO_DIR = "ToneFit"

if os.path.isdir(REPO_DIR):
    print("Repo already cloned. Pulling latest changes...")
    !cd {REPO_DIR} && git pull
else:
    print("Cloning repo...")
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

In [ ]:
# Install all required packages
# (tensorflow and scikit-learn are already on Colab, but we pin versions to be safe)
!pip install -q \
    icrawler \
    imagehash \
    scikit-image \
    opencv-python-headless

print("All dependencies installed.")

---
## Step 2 — Data Collection

**Two options — pick one:**
- **Option A (recommended):** Run `collect_data.py` to scrape Google Images automatically
- **Option B (manual):** Upload your own `dataset/` folder if you already have images

Run Option A first. If scraping fails or gives too few images, switch to Option B.

In [ ]:
# ── Option A: Automatic scraping ──────────────────────────────────────────────
#
# This downloads ~5 images per celebrity, detects faces, and saves 224x224 crops.
# Takes ~30–40 minutes for all 4 seasons. You can split by season if needed:
#   Change RUN_SEASONS = ["spring"] to run only one season.
#
# If you want to run only specific seasons, edit collect_data.py in the repo
# or override RUN_SEASONS below before importing.

import importlib, sys

# Reload in case the module was already imported
if 'collectdata' in sys.modules:
    del sys.modules['collectdata']

import collectdata

# To run all seasons:
collectdata.run(seasons_to_run=None)

# To run just one season (comment out the line above and uncomment below):
# collectdata.run(seasons_to_run=["spring"])

In [ ]:
# Check how many images were collected
import os
seasons = ["spring", "summer", "autumn", "winter"]
total = 0
print("Dataset summary:")
for s in seasons:
    folder = os.path.join("dataset", s)
    if os.path.isdir(folder):
        count = len([f for f in os.listdir(folder)
                     if f.lower().endswith((".jpg",".jpeg",".png",".webp",".bmp"))])
        total += count
        bar = "█" * (count // 3)
        print(f"  {s:<8}: {count:>4} images  {bar}")
    else:
        print(f"  {s:<8}: folder not found")
print(f"  TOTAL   : {total} images")

In [ ]:
# ── Option B: Upload your own dataset ─────────────────────────────────────────
#
# If you already have face images ready, upload a ZIP of your dataset/ folder.
# Structure should be:
#   dataset/spring/  ← face crop .jpg files
#   dataset/summer/
#   dataset/autumn/
#   dataset/winter/
#
# Uncomment and run the cell below to upload.

# from google.colab import files
# import zipfile
#
# print("Select your dataset ZIP file to upload...")
# uploaded = files.upload()
# zip_name = list(uploaded.keys())[0]
#
# with zipfile.ZipFile(zip_name, 'r') as z:
#     z.extractall(".")
# print("Dataset extracted.")

print("Option B cell is ready. Uncomment and run if needed.")

---
## Step 3 — Preprocessing

Removes duplicates, extracts CIELab + HSV color features, normalizes, and splits the dataset.

In [ ]:
if 'preprocess' in sys.modules:
    del sys.modules['preprocess']

import preprocess
preprocess.run()

In [ ]:
# Verify outputs
import numpy as np
import pandas as pd

df       = pd.read_csv("features.csv")
X_train  = np.load("X_train.npy")
X_test   = np.load("X_test.npy")
y_train  = np.load("y_train.npy")
y_test   = np.load("y_test.npy")

print(f"features.csv  : {len(df)} rows, {len(df.columns)} columns")
print(f"X_train shape : {X_train.shape}")
print(f"X_test  shape : {X_test.shape}")
print(f"y_train shape : {y_train.shape}")
print(f"y_test  shape : {y_test.shape}")
print("\nClass distribution in features.csv:")
print(df["season"].value_counts())

---
## Step 4 — Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib
matplotlib.use('Agg')   # avoid display issues on Colab
import matplotlib.pyplot as plt
%matplotlib inline

os.makedirs("results", exist_ok=True)
print("results/ folder ready")

In [ ]:
# ── 4a. Class distribution ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

SEASONS = ["spring", "summer", "autumn", "winter"]
SEASON_COLORS = {
    "spring": "#F4A261",
    "summer": "#90B4CE",
    "autumn": "#A0522D",
    "winter": "#5B5EA6",
}

counts = df["season"].value_counts().reindex(SEASONS)
colors = [SEASON_COLORS[s] for s in SEASONS]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(SEASONS, counts.values, color=colors, edgecolor="black", linewidth=0.7)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+1,
            str(count), ha="center", fontsize=11, fontweight="bold")
ax.set_title("Class Distribution — Images per Season", fontsize=14, fontweight="bold")
ax.set_xlabel("Season"); ax.set_ylabel("Images")
ax.axhline(counts.mean(), color="gray", linestyle="--", linewidth=1,
           label=f"Mean: {counts.mean():.0f}")
ax.legend(); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("results/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/class_distribution.png")

In [ ]:
# ── 4b. Sample face images per season ─────────────────────────────────────────
import cv2
SAMPLES = 5
VALID_EXT = (".jpg",".jpeg",".png",".webp",".bmp")

fig, axes = plt.subplots(len(SEASONS), SAMPLES, figsize=(SAMPLES*2.2, len(SEASONS)*2.5))

for row, season in enumerate(SEASONS):
    season_dir = os.path.join("dataset", season)
    files = sorted([f for f in os.listdir(season_dir)
                    if f.lower().endswith(VALID_EXT)])[:SAMPLES] if os.path.isdir(season_dir) else []
    for col in range(SAMPLES):
        ax = axes[row][col]
        ax.axis("off")
        if col < len(files):
            img = cv2.imread(os.path.join(season_dir, files[col]))
            if img is not None:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if col == 0:
            ax.set_ylabel(season.capitalize(), fontsize=12, fontweight="bold",
                          color=SEASON_COLORS[season], rotation=90, labelpad=10)
            ax.yaxis.set_label_coords(-0.15, 0.5)

fig.suptitle("Sample Face Crops per Season", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("results/sample_images.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/sample_images.png")

In [ ]:
# ── 4c. Feature distributions — L*, a*, b*, ITA ───────────────────────────────
import seaborn as sns

BOX_FEATURES = [
    ("L_mean", "L* Mean (Lightness)"),
    ("a_mean", "a* Mean (Red-Green)"),
    ("b_mean", "b* Mean (Yellow-Blue / Warmth)"),
    ("ITA",    "ITA Score (Skin Tone Angle)"),
]
palette = [SEASON_COLORS[s] for s in SEASONS]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

for i, (feat, label) in enumerate(BOX_FEATURES):
    ax = axes[i]
    data = [df.loc[df["season"]==s, feat].values for s in SEASONS]
    bp = ax.boxplot(data, patch_artist=True,
                    medianprops=dict(color="black", linewidth=2))
    for patch, col in zip(bp["boxes"], palette):
        patch.set_facecolor(col); patch.set_alpha(0.75)
    ax.set_xticklabels([s.capitalize() for s in SEASONS], fontsize=11)
    ax.set_title(label, fontsize=12, fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)

fig.suptitle("Feature Distributions per Season", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results/feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/feature_distributions.png")

In [ ]:
# ── 4d. Correlation heatmap ───────────────────────────────────────────────────
FEATURE_COLS = ["L_mean","a_mean","b_mean","L_std","a_std","b_std",
                "ITA","H_mean","S_mean","V_mean"]
corr = df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={"size":9}, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/correlation_heatmap.png")

In [ ]:
# ── 4e. PCA visualization ─────────────────────────────────────────────────────
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

X_pca_raw = df[FEATURE_COLS].values.astype("float32")
X_pca_scaled = MinMaxScaler().fit_transform(X_pca_raw)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_pca_scaled)
explained = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(9, 7))
for season in SEASONS:
    mask = df["season"] == season
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=SEASON_COLORS[season], label=season.capitalize(),
               alpha=0.7, edgecolors="white", linewidths=0.4, s=60)

ax.set_xlabel(f"PC1 ({explained[0]:.1f}% variance)", fontsize=12)
ax.set_ylabel(f"PC2 ({explained[1]:.1f}% variance)", fontsize=12)
ax.set_title("PCA — Personal Color Seasons in 2D", fontsize=13, fontweight="bold")
ax.legend(fontsize=11); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("results/pca_visualization.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"PC1: {explained[0]:.1f}%  PC2: {explained[1]:.1f}%  Total: {sum(explained):.1f}%")
print("Saved: results/pca_visualization.png")

---
## Step 5 — Train Traditional ML (SVM + Random Forest)

In [ ]:
if 'train_traditional' in sys.modules:
    del sys.modules['train_traditional']

import train_traditional
train_traditional.run()

In [ ]:
# Verify models were saved
for f in ["models/svm_model.pkl", "models/rf_model.pkl", "models/scaler.pkl"]:
    size_kb = os.path.getsize(f) / 1024 if os.path.exists(f) else 0
    status  = f"{size_kb:.1f} KB" if os.path.exists(f) else "NOT FOUND"
    print(f"  {f:<35}: {status}")

---
## Step 6 — Train Deep Learning (MobileNetV2)

Make sure GPU is enabled (`Runtime → Change runtime type → T4 GPU`).
Phase 1 trains ~30 epochs, Phase 2 fine-tunes ~20 epochs — expect ~10–20 minutes on GPU.

In [ ]:
if 'train_deeplearning' in sys.modules:
    del sys.modules['train_deeplearning']

import train_deeplearning
train_deeplearning.run()

In [ ]:
# Show training curves
from IPython.display import Image as IPImage, display
display(IPImage("results/mobilenetv2_training_curves.png"))

In [ ]:
# Verify model was saved
dl_path = "models/mobilenetv2_model.h5"
if os.path.exists(dl_path):
    size_mb = os.path.getsize(dl_path) / (1024*1024)
    print(f"  {dl_path}: {size_mb:.1f} MB")
else:
    print(f"  {dl_path}: NOT FOUND")

---
## Step 7 — Evaluate All Models

In [ ]:
if 'evaluate' in sys.modules:
    del sys.modules['evaluate']

import evaluate
evaluate.run()

In [ ]:
# Show comparison table
import pandas as pd
results_df = pd.read_csv("results/comparison_table.csv")
display(results_df)

In [ ]:
# Show combined confusion matrices
from IPython.display import Image as IPImage, display
display(IPImage("results/confusion_matrices_combined.png"))

In [ ]:
# Show model comparison bar chart
display(IPImage("results/model_comparison_chart.png"))

---
## Step 8 — Predict on a Test Photo

Upload any face photo to test the prediction demo.

In [ ]:
# Upload a face photo to test
from google.colab import files

print("Click 'Choose Files' and upload a face photo (jpg or png)...")
uploaded = files.upload()

if uploaded:
    test_image_path = list(uploaded.keys())[0]
    print(f"Uploaded: {test_image_path}")
else:
    print("No file uploaded.")

In [ ]:
# Run prediction on the uploaded photo
if 'predict' in sys.modules:
    del sys.modules['predict']

import predict

# show_display=False because Colab uses inline matplotlib — the saved image is shown below
predict.run(test_image_path, show_display=False)

In [ ]:
# Show the result image inline
from IPython.display import Image as IPImage, display
import os

img_name    = os.path.splitext(os.path.basename(test_image_path))[0]
result_path = f"results/prediction_{img_name}.png"

if os.path.exists(result_path):
    display(IPImage(result_path))
else:
    print(f"Result image not found at {result_path}")

---
## Step 9 — Download Results

Download everything (models + result plots) as a ZIP to save to your computer.

In [ ]:
import zipfile
from google.colab import files
import glob

zip_path = "ToneFit_results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    # Results (plots + CSV)
    for f in glob.glob("results/*"):
        z.write(f)
    # Trained models
    for f in glob.glob("models/*"):
        z.write(f)
    # Feature CSV
    for fname in ["features.csv", "X_train.npy", "X_test.npy",
                  "y_train.npy", "y_test.npy"]:
        if os.path.exists(fname):
            z.write(fname)

size_mb = os.path.getsize(zip_path) / (1024*1024)
print(f"Created: {zip_path} ({size_mb:.1f} MB)")
print("Downloading...")
files.download(zip_path)